In [22]:
import base64

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

img_b64 = encode_image(image_path)

In [21]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Uploaded:", image_path)

Saving sample_feedback.jpg to sample_feedback (1).jpg
Uploaded: sample_feedback (1).jpg


In [1]:
!pip install requests pillow

In [12]:
# import requests
# import json

# MISTRAL_API_KEY = "pQrDCyZweXMvNuIgf6JXmEeIOouNeuCb"

# def analyze_image_with_vlm(image_b64):
#     url = "https://api.mistral.ai/v1/chat/completions"

#     headers = {
#         "Authorization": f"Bearer {MISTRAL_API_KEY}",
#         "Content-Type": "application/json"
#     }

#     prompt = """
# You are an AI system that reads handwritten documents.

# TASK:
# 1. Understand the image content directly (no OCR step).
# 2. Extract sentiment.

# Return ONLY valid JSON:

# {
#   "positive": [],
#   "negative": [],
#   "aspects": {
#     "supervisor": {"positive": [], "negative": []},
#     "environment": {"positive": [], "negative": []},
#     "tasks": {"positive": [], "negative": []},
#     "communication": {"positive": [], "negative": []},
#     "workload": {"positive": [], "negative": []}
#   }
# }

# Rules:
# - Only full meaningful sentences or clauses
# - No single words
# - No explanations
# """

#     payload = {
#         "model": "pixtral-12b",  # if available on your account
#         "messages": [
#             {
#                 "role": "user",
#                 "content": [
#                     {"type": "text", "text": prompt},
#                     {
#                         "type": "image_url",
#                         "image_url": f"data:image/png;base64,{image_b64}"
#                     }
#                 ]
#             }
#         ],
#         "temperature": 0
#     }

#     response = requests.post(url, headers=headers, json=payload)

#     return response.json()["choices"][0]["message"]["content"]

In [19]:
result = analyze_image_with_vlm(img_b64)
print(result)

```json
{
  "positive": [
    "I want to learn and implement a full AI system solution",
    "Interested with AI/blockchain solution",
    "To train on a fresh solution in front of the ground"
  ],
  "negative": [],

  "aspects": {
    "supervisor": {"positive": [], "negative": []},
    "environment": {"positive": [], "negative": []},
    "tasks": {
      "positive": [
        "Learn how to deploy a scalable system",
        "Deal with a million customers",
        "Assure security of this system"
      ],
      "negative": []
    },
    "communication": {"positive": [], "negative": []},
    "workload": {"positive": [], "negative": []}
  }
}
```


In [23]:
import requests

MISTRAL_API_KEY = "pQrDCyZweXMvNuIgf6JXmEeIOouNeuCb"

def extract_text_vlm(image_b64):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = """
You are an OCR system.

TASK:
Extract ONLY the exact text written in the image.

RULES:
- Do NOT analyze sentiment
- Do NOT summarize
- Do NOT interpret
- Keep original wording

Return ONLY JSON:

{
  "text": "..."
}
"""

    payload = {
        "model": "pixtral-12b",
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": f"data:image/png;base64,{image_b64}"
                    }
                ]
            }
        ],
        "temperature": 0
    }

    r = requests.post(url, headers=headers, json=payload)
    return r.json()["choices"][0]["message"]["content"]

In [24]:
raw = extract_text_vlm(img_b64)
print(raw)

```json
{
  "text": "- by the end of the program I'm hopeful and want to learn about all those complex concepts, enrich my knowledge and most important, be able to develop the capacity of being able to explain an idea clearly from start to finish. I definitely need to work on that (learn how to communicate my thoughts)
- I expect to be able to understand most things about AI, blockchains etc - be less timid and be a member that the group can rely on"
}
```


In [31]:
def analyze_text_vlm(text):
    url = "https://api.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""
You are a sentiment analysis system.

INPUT TEXT:
{text}

TASK:
Extract:
- positive statements
- negative statements

RULES:
- Only full sentences or clauses
- No single words
- Return ONLY JSON

FORMAT:
{{
  "positive": [],
  "negative": [],

  }}
}}
"""

    payload = {
        "model": "mistral-small-latest",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }

    r = requests.post(url, headers=headers, json=payload)
    return r.json()["choices"][0]["message"]["content"]

In [32]:
import json
import re

text_json = extract_text_vlm(img_b64)
print("STEP 1:", text_json)

# Remove markdown code block fences before parsing
clean_text_json = text_json.strip('```json\n').strip('\n```')

# The model sometimes returns JSON where string values contain unescaped newlines.
# JSON requires newlines within string literals to be escaped as \\n.
# Use regex to find the content of the 'text' field and escape its newlines.
match = re.match(r'\{\s*"text":\s*"(?P<text_content>.*?)"\s*\}', clean_text_json, re.DOTALL)

if match:
    text_content = match.group('text_content')
    # Replace unescaped newlines with escaped ones for JSON parsing
    escaped_text_content = text_content.replace('\n', '\\n')
    # Reconstruct the JSON string with the escaped content
    repaired_json = clean_text_json.replace(text_content, escaped_text_content)
    text = json.loads(repaired_json)["text"]
else:
    # Fallback if the regex doesn't match (should not happen with current model output)
    print("Warning: Could not parse text content using regex. Attempting direct JSON load.")
    text = json.loads(clean_text_json)["text"]

analysis = analyze_text_vlm(text)
print("STEP 2:", analysis)

STEP 1: ```json
{
  "text": "- by the end of the program I'm hopeful and want to learn about all those complex concepts, enrich my knowledge and most important, be able to develop the capacity of being able to explain an idea clearly from start to finish. I definitely need to work on that (learn how to communicate my thoughts)
- I expect to be able to understand most things about AI, blockchains etc - be less timid and be a member that the group can rely on"
}
```
STEP 2: ```json
{
  "positive": [
    "by the end of the program I'm hopeful and want to learn about all those complex concepts",
    "enrich my knowledge",
    "be able to develop the capacity of being able to explain an idea clearly from start to finish",
    "I expect to be able to understand most things about AI, blockchains etc",
    "be a member that the group can rely on"
  ],
  "negative": [
    "I definitely need to work on that (learn how to communicate my thoughts)",
    "be less timid"
  ]
}
```
